In [10]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
for _pkg in ('plotly', 'anywidget'):        # anywidget backs Plotly's FigureWidget
    try:
        __import__(_pkg)
    except ImportError:
        import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
import towers as tw, widgets as wg

INSTANCE      = 'results/sukhumvit_90.json'
QPU_RESULTS   = 'results/headline_qpu_90_results.json'
BASEMAP_PNG   = 'data/sukhumvit_basemap.png'
BASEMAP_JSON  = 'data/sukhumvit_basemap.json'

inst = tw.load_instance(INSTANCE)
res  = tw.load_results(QPU_RESULTS)
sel, ret, tot, defect = tw.decode_shots(res['shots'], inst['n_atoms'], policy='postselect')
mis, viol, nvalid = tw.best_valid_set(sel, inst['graph'])   # reject blockade violations
print(f"{res['source']}  |  {tot} shots  |  post-selected {ret}  |  {nvalid} valid independent sets")
print(f"largest co-channel group: {len(mis)} of {inst['n_atoms']} towers share one channel")

QPU  |  1000 shots  |  post-selected 563  |  97 valid independent sets
largest co-channel group: 34 of 90 towers share one channel


## Is the result a genuine MIS?
Before we display it, prove the returned graph state is a real independent set — the coverage certificate.

In [11]:
G = inst['graph']; S = set(mis)
bad = [(u, v) for u, v in G.edges() if u in S and v in S]
addable = [v for v in G if v not in S and not any(nb in S for nb in G.neighbors(v))]
print(f"co-channel group size: {len(S)}   (exact optimum {inst['classical']['exact_size']})")
print(f"independent set      : {'YES - 0 conflicting pairs' if not bad else f'NO ({len(bad)} violations)'}")
print(f"maximal              : {'YES - no tower can be added' if not addable else f'NO - could add {addable}'}")
print(f"reuse certificate    : {'INTERFERENCE-FREE' if tw.verify_independent_set(G, S) else 'FAILED'}")

co-channel group size: 34   (exact optimum 35)
independent set      : YES - 0 conflicting pairs
maximal              : YES - no tower can be added
reuse certificate    : INTERFERENCE-FREE


## ①  From district to atoms
Step through the three states with the toggle.

In [ ]:
wg.pipeline_view(inst, BASEMAP_PNG, BASEMAP_JSON)

## ②  Frequency reuse — one MIS per channel

Every channel was **measured on QuEra Aquila** (iterated MIS on the residual sub-register). Step through the channels — each MIS is one frequency; the conflict edges thin out until the whole district runs on a handful of channels.

In [ ]:
import json
assign = json.load(open('results/channel_assignment_90_qpu.json'))   # real QPU coloring
edges  = list(inst['graph'].edges())
print(f"{assign['source']}: {assign['n_channels']} channels "
      f"(optimal {assign['chromatic_number']}, greedy overspend {assign['overspend']}); "
      f"round sizes {assign['round_sizes']}")
wg.channel_stepper_view(assign, edges, BASEMAP_PNG, BASEMAP_JSON)

## Next steps: the qBraid Vault Challenge

Interested in working live with other users on qBraid? Come join the **Vault Challenge**: work alongside the qBraid community and stand a chance to win prizes.

**Thirteen quantum vaults. Two weeks. One question: can you unscramble them?** Each vault hides a circuit that scrambles a register of qubits away from all-zeros; your job is to write a circuit that runs after it and drives the system back to `|0…0⟩`. The catch: a perfect solution is not automatically the best one. Your score is the probability of landing on all-zeros, scaled down by every two-qubit gate you spend getting there.

**Prizes**
- 🥇 $100 Amazon gift card + 10,000 qBraid credits
- 🥈 $50 + 5,000 credits
- 🥉 $25 + 2,500 credits
- Credits pay for real QPU time, GPU compute, and AI usage with leading models.

👉 **Join the challenge:** https://account.qbraid.com/explore?tab=challenges